# IADD Traffic Detection - YOLOv11 (Run 3, eval v3-2)

Re-evaluation of the already-trained Run-3 model (yolo11s, imgsz 960). No training
happens here - it only loads `run3/weights/best.pt` and regenerates the evaluation
outputs in the same format as Run 4:
- mAP at the default threshold (averaged over all confidences -> the honest number).
- A clean confusion matrix at Run-3's own F1-optimal confidence, read from its val
  F1 curve (not hard-coded), so the two runs are comparable.

Prerequisite: `iadd_runs/run3/weights/best.pt` must be on Drive.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Unzip the dataset into Colab

Colab's local disk is wiped on reconnect, so re-run this cell after any reconnect.

In [ ]:
!rm -rf /content/iadd_subset_v3
!unzip -q "/content/drive/MyDrive/iadd_subset_v3.zip" -d /content
!ls /content/iadd_subset_v3

## 5. Validate, then pick the operating confidence

Standard validation of Run-3's best checkpoint (COCO-standard mAP + per-class table
+ P/R/F1/PR curves). Then read the confidence that maximises mean F1 from the val F1
curve, and render a clean confusion matrix at that point in `run3_val_tuned`.
Ultralytics validates at the model's training resolution (960) automatically.

In [ ]:
from ultralytics import YOLO
import numpy as np

DATA = '/content/iadd_subset_v3/data.yaml'
RUNS = '/content/drive/MyDrive/iadd_runs'
best = YOLO(f'{RUNS}/run3/weights/best.pt')

# Honest metrics: mAP is averaged over all thresholds, so it is not tied to any conf.
mv = best.val(data=DATA, split='val', plots=True,
              project=RUNS, name='run3_val', exist_ok=True)
print('VAL  mAP50=%.4f  mAP50-95=%.4f' % (mv.box.map50, mv.box.map))

# Operating point = confidence with the highest mean F1, read from the val F1 curve.
CONF = 0.25
for x, y, _, ylabel in mv.curves_results:
    if ylabel == 'F1':
        px = np.asarray(x)
        y = np.asarray(y)
        f1 = y.mean(0) if y.ndim > 1 else y
        CONF = round(float(px[f1.argmax()]), 3)
print('F1-optimal confidence (from val) =', CONF)

# Clean confusion matrix + per-class table at that operating point.
best.val(data=DATA, split='val', conf=CONF, plots=True,
         project=RUNS, name='run3_val_tuned', exist_ok=True)
print('clean val plots -> run3_val_tuned  (conf=%.3f)' % CONF)

## 6. Test - honest mAP, and a comparable confusion matrix

The test videos were never seen in training (leakage-free split). The second call
uses the same operating confidence, so its confusion matrix is comparable to Run 4's.

In [ ]:
mt = best.val(data=DATA, split='test', plots=True,
              project=RUNS, name='run3_test', exist_ok=True)
print('TEST mAP50=%.4f  mAP50-95=%.4f' % (mt.box.map50, mt.box.map))

best.val(data=DATA, split='test', conf=CONF, plots=True,
         project=RUNS, name='run3_test_tuned', exist_ok=True)
print('clean test plots -> run3_test_tuned  (conf=%.3f)' % CONF)

## 7. Demo predictions - saved to run3/demo

Runs the model on a few test images at the operating confidence and writes the
annotated images to a folder on Drive. Nothing is displayed inline.

In [ ]:
import glob

imgs = sorted(glob.glob('/content/iadd_subset_v3/images/test/*.jpg'))[:16]
best.predict(imgs, conf=CONF, save=True,
             project=f'{RUNS}/run3', name='demo', exist_ok=True)
print('demo images saved -> iadd_runs/run3/demo  (conf=%.3f)' % CONF)

---
Results on Drive under `iadd_runs/`:
- `run3_val/`, `run3_test/` - honest metrics + curves (default threshold)
- `run3_val_tuned/`, `run3_test_tuned/` - clean confusion matrix at the F1-optimal conf
- `run3/demo/` - annotated demo images

Note: this overwrites the previous `run3_val_tuned` / `run3_test_tuned` folders on
Drive (the old ones used Run-3's 0.434); the local copies under colab/runs are untouched.